In [ ]:
import numpy as np
import os
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from typing import List, TypeVar
import pywt
Tensor = TypeVar('torch.tensor')

is_norm = True
new_min, new_max = -1, 1

input_dim = 15
hidden_dim = [64,128]

seq_len = 150
latent_dim = 128

In [ ]:
def data_to_model(data, sequence_length=120, step_size=25):
    data = data.float()
    sequences = []
    N = data.shape[0]
    
    for start_idx in range(0, N - sequence_length + 1, step_size):
        sequence = data[start_idx:start_idx + sequence_length]
        sequences.append(sequence)
    
    sequences = torch.stack(sequences)
    return sequences

def get_data(mode = '1-hover', phase = '2_5'):
    # 1-hover 2-forward 3-acc-x-axis-flag3-3000 5-motor03-flag4-085 6-motor03-flag3-200
    fmode = mode
    base_path = os.getcwd().split('\model')[0]

    sycn_raw_data = f'all_sycn_raw_data_mode{phase}.csv'
    sycn_err_data = f'all_sycn_err_data_mode{phase}.csv'
    sycn_sim_data = f'all_sycn_sim_data_mode{phase}.csv'

    sycn_raw_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_raw_data)
    sycn_err_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_err_data)
    sycn_sim_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_sim_data)

    raw_data = pd.read_csv(sycn_raw_data_path).to_numpy()
    err_data = pd.read_csv(sycn_err_data_path).to_numpy()
    sim_data = pd.read_csv(sycn_sim_data_path).to_numpy()

    return raw_data, sim_data, err_data

def get_train_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3']):
    fi_raw, fi_sim, fi_err = get_data(mode= mode[0], phase=phase[0])
    sim_data = np.vstack((fi_sim))
    raw_data = np.vstack((fi_raw))
    err_data = np.vstack((fi_err))
    for mo, ph in zip(mode[1:], phase[1:]):
        raw, sim, err = get_data(mode= mo, phase= ph)
        sim_data = np.vstack((sim_data, sim))
        raw_data = np.vstack((raw_data, raw))
        err_data = np.vstack((err_data, err))

    sim_data_split = np.split(sim_data, indices_or_sections=6, axis=1)
    raw_data_split = np.split(raw_data, indices_or_sections=6, axis=1)
    err_data_split = np.split(err_data, indices_or_sections=6, axis=1)

    sim_gyro, sim_acc, sim_mag, sim_pos, sim_vel, sim_eacc = sim_data_split[0], sim_data_split[1], sim_data_split[2], sim_data_split[3], sim_data_split[4], sim_data_split[5]
    raw_gyro, raw_acc, raw_mag, raw_pos, raw_vel, raw_eacc = raw_data_split[0], raw_data_split[1], raw_data_split[2], raw_data_split[3], raw_data_split[4], raw_data_split[5]
    err_gyro, err_acc, err_mag, err_pos, err_vel, err_eacc = err_data_split[0], err_data_split[1], err_data_split[2], err_data_split[3], err_data_split[4], err_data_split[5]

    sim_data_ = np.hstack((sim_acc, sim_gyro, sim_mag, sim_pos, sim_vel))
    raw_data_ = np.hstack((raw_acc, raw_gyro, raw_mag, raw_pos, raw_vel))
    err_data_ = np.hstack((err_acc, err_gyro, err_mag, err_pos, err_vel))

    return torch.tensor(sim_data_), torch.tensor(raw_data_), torch.tensor(err_data_)

def get_sensor_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3'], label = 1, shuffle=False, ratio=1, sequence_length=80, step_size=1):
    sim, raw, err = get_train_data(mode, phase)

    if is_norm:
        sim_min_vals = torch.min(sim, dim=0)[0] 
        sim_max_vals = torch.max(sim, dim=0)[0]
        sim_normalized = ((sim - sim_min_vals) / (sim_max_vals - sim_min_vals)) * (new_max - new_min) + new_min

        raw_min_vals = torch.min(raw, dim=0)[0] 
        raw_max_vals = torch.max(raw, dim=0)[0]
        raw_normalized = ((raw - raw_min_vals) / (raw_max_vals - raw_min_vals)) * (new_max - new_min) + new_min

        err_min_vals = torch.min(err, dim=0)[0] 
        err_max_vals = torch.max(err, dim=0)[0]
        err_normalized = ((err - err_min_vals) / (err_max_vals - err_min_vals)) * (new_max - new_min) + new_min
    else:
        sim_min_vals, sim_max_vals, raw_min_vals, raw_max_vals, err_min_vals, err_max_vals = None, None, None, None, None, None
        sim_normalized, raw_normalized, err_normalized = sim, raw, err

    sim_2_model = data_to_model(data=sim_normalized, sequence_length=sequence_length, step_size=step_size)
    sim_2_model_label = torch.full((sim_2_model.size(0), 1), label, dtype=torch.long)

    raw_2_model = data_to_model(data=raw_normalized, sequence_length=sequence_length, step_size=step_size)
    raw_2_model_label = torch.full((raw_2_model.size(0), 1), label, dtype=torch.long)

    err_2_model = data_to_model(data=err_normalized, sequence_length=sequence_length, step_size=step_size)
    err_2_model_label = torch.full((err_2_model.size(0), 1), label, dtype=torch.long)

    if shuffle:
        torch.manual_seed(42)
        indices = torch.randperm(sim_2_model.size(0))

        sim_2_model = sim_2_model[indices]
        sim_2_model_label = sim_2_model_label[indices]
        raw_2_model = raw_2_model[indices]
        raw_2_model_label = raw_2_model_label[indices]
        err_2_model = err_2_model[indices]
        err_2_model_label = err_2_model_label[indices]

    sim_2_model = sim_2_model[:int(sim_2_model.size(0) * ratio)]
    sim_2_model_label = sim_2_model_label[:int(sim_2_model_label.size(0) * ratio)]
    raw_2_model = raw_2_model[:int(raw_2_model.size(0) * ratio)]
    raw_2_model_label = raw_2_model_label[:int(raw_2_model_label.size(0) * ratio)]
    err_2_model = err_2_model[:int(err_2_model.size(0) * ratio)]
    err_2_model_label = err_2_model_label[:int(err_2_model_label.size(0) * ratio)]

    return sim_2_model, sim_2_model_label, raw_2_model, raw_2_model_label, err_2_model, err_2_model_label, (raw_min_vals, raw_max_vals), (sim_min_vals, sim_max_vals), (err_min_vals, err_max_vals)

def normalize_data(x, data_min, data_max):
    return (x - data_min) / (data_max - data_min)

def denormalize_data(x, data_min, data_max):
    return ((x - new_min) / (new_max - new_min)) * (data_max - data_min) + data_min

def classify_features_tensor(data, wavelet='db4', level=3, threshold=0.9):
    assert len(data.shape) == 3, 
    
    batch, seq_len, features = data.shape
    classification = []
    high_freq_indices = []
    low_freq_indices = []
    
    for f in range(features):
        feature_data = data[:, :, f] 
        low_energy_total = 0
        high_energy_total = 0
        
        for b in range(batch):
            feature_np = feature_data[b].detach().cpu().numpy()  # 转为 numpy
            coeffs = pywt.wavedec(feature_np, wavelet=wavelet, level=level)
            
            cA = coeffs[0]
            cD_energy = sum(np.sum(c**2) for c in coeffs[1:]) 
            cA_energy = np.sum(cA**2) 
            
            low_energy_total += cA_energy
            high_energy_total += cD_energy
        
        low_energy_ratio = low_energy_total / (low_energy_total + high_energy_total)
        
        if low_energy_ratio > threshold:
            classification.append("Low-Frequency")  
            low_freq_indices.append(f)
        else:
            classification.append("High-Frequency") 
            high_freq_indices.append(f)
    
    return classification, low_freq_indices, high_freq_indices

# 1-hover 2-forward 3-acc-x-axis-flag3-3000 5-motor03-flag4-085 6-motor03-flag3-200
hover_2_model_sim, hover_2_model_sim_lables, hover_2_model_raw, hover_2_model_raw_lables, hover_2_model_err, hover_2_model_err_lables, (hover_raw_min_vals, hover_raw_max_vals), (hover_sim_min_vals, hover_sim_max_vals), (hover_err_min_vals, hover_err_max_vals) = get_sensor_data(mode= ['1-hover'], phase = ['2_6'], label = 1, shuffle=True, ratio=1, sequence_length=seq_len, step_size=4)
takeoff_2_model_sim, takeoff_2_model_sim_lables, takeoff_2_model_raw, takeoff_2_model_raw_lables, takeoff_2_model_err, takeoff_2_model_err_lables, (takeoff_raw_min_vals, takeoff_raw_max_vals), (takeoff_sim_min_vals, takeoff_sim_max_vals), (takeoff_err_min_vals, takeoff_err_max_vals) = get_sensor_data(mode= ['1-hover'], phase = ['2_3'], label = 2, shuffle=True, ratio=1, sequence_length=seq_len, step_size=4)

err_2_model = torch.cat((hover_2_model_err, takeoff_2_model_err), dim=0)
raw_2_model = torch.cat((hover_2_model_raw, takeoff_2_model_raw), dim=0)
sim_2_model = torch.cat((hover_2_model_sim, takeoff_2_model_sim), dim=0)
label_2_model = torch.cat((hover_2_model_raw_lables, takeoff_2_model_raw_lables), dim=0)

feature_classification, low_freq_indices, high_freq_indices = classify_features_tensor(err_2_model, wavelet='db4', level=3, threshold=0.8)
print(f"Feature: {feature_classification}")
print(f'low_freq_indices:{low_freq_indices}')
print(f'high_freq_indices:{high_freq_indices}')

high_recon_dim = [64,len(high_freq_indices)]
low_recon_dim = [64,len(low_freq_indices)]

label_ana = {
    '1': (hover_raw_min_vals, hover_raw_max_vals, hover_sim_min_vals, hover_sim_max_vals, hover_err_min_vals, hover_err_max_vals),
    '2':(takeoff_raw_min_vals, takeoff_raw_max_vals, takeoff_sim_min_vals, takeoff_sim_max_vals, takeoff_err_min_vals, takeoff_err_max_vals)
}

print(f'hover_2_model_sim:{hover_2_model_sim.shape} takeoff_2_model_sim:{takeoff_2_model_sim.shape}')
print(f'err_2_model:{err_2_model.shape} label_2_model:{label_2_model.shape}')

dataset = TensorDataset(err_2_model, raw_2_model, sim_2_model, label_2_model)
batch_size = 4  
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
sh = next(iter(dataloader))[0].shape
print(f'dataloader Len: {len(dataloader)} dataloader Shape: {sh}')


In [ ]:
class MVAE(nn.Module):

    def __init__(self,
                 low_in_channels: int,  
                 high_in_channels: int,
                 latent_dim: int, 
                 hidden_dims: List = None,  
                 high_recon_dims: List = None,
                 low_recon_dims: List = None,
                 **kwargs) -> None:
        super(MVAE, self).__init__()

        self.latent_dim = latent_dim

        if hidden_dims is None:
            hidden_dims = [128, 256, 512]

        self.low_freq_encoder = nn.ModuleList()
        input_dim = low_in_channels 
        for h_dim in hidden_dims:
            self.low_freq_encoder.append(nn.LSTM(input_dim, h_dim, batch_first=True))
            input_dim = h_dim

        self.high_freq_encoder = nn.ModuleList()
        input_dim = high_in_channels 
        self.high_freq_encoder_input = nn.Sequential(
            nn.Conv1d(input_dim, input_dim, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU())
        for h_dim in hidden_dims:
            self.high_freq_encoder.append(nn.LSTM(input_dim, h_dim, batch_first=True))
            input_dim = h_dim

        self.fc_mu_low = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var_low = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_mu_high = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var_high = nn.Linear(hidden_dims[-1], latent_dim)

        self.low_freq_decoder = nn.ModuleList()
        input_dim = latent_dim
        for h_dim in low_recon_dims:
            self.low_freq_decoder.append(nn.LSTM(input_dim, h_dim, batch_first=True))
            input_dim = h_dim

        self.high_freq_decoder = nn.ModuleList()
        input_dim = latent_dim
        self.high_freq_decode_input = nn.Sequential(
            nn.Conv1d(input_dim, input_dim, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU())
        for h_dim in high_recon_dims:
            self.high_freq_decoder.append(nn.LSTM(input_dim, h_dim, batch_first=True))
            input_dim = h_dim

    def encode_low_freq(self, input: Tensor) -> Tensor:
        x = input
        for lstm in self.low_freq_encoder:
            x, _ = lstm(x)
        return x

    def encode_high_freq(self, input: Tensor) -> Tensor:
        x = input.permute(0, 2, 1) # (batch, channels, seq_len)
        x = self.high_freq_encoder_input(x)
        x = x.permute(0, 2, 1)
        for lstm in self.high_freq_encoder:
            x, _ = lstm(x)
        return x

    def decode_low_freq(self, input: Tensor) -> Tensor:
        x = input
        for lstm in self.low_freq_decoder:
            x, _ = lstm(x)
        return x

    def decode_high_freq(self, input: Tensor) -> Tensor:
        x = input.permute(0, 2, 1) # (batch, channels, seq_len)
        x = self.high_freq_decode_input(x)
        x = x.permute(0, 2, 1)
        for lstm in self.high_freq_decoder:
            x, _ = lstm(x)
        return x

    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(mu)
        return eps * std + mu

    def forward(self, input: Tensor, high_freq_indices: List[int], low_freq_indices: List[int]) -> Tensor:
        high_freq_input = input[:, :, high_freq_indices]
        low_freq_input = input[:, :, low_freq_indices]

        low_freq_encoded = self.encode_low_freq(low_freq_input)
        mu_low_freq = self.fc_mu_low(low_freq_encoded)
        log_var_low_freq = self.fc_var_low(low_freq_encoded)
        z_low_freq = self.reparameterize(mu_low_freq, log_var_low_freq)

        high_freq_encoded = self.encode_high_freq(high_freq_input)
        mu_high_freq = self.fc_mu_high(high_freq_encoded)
        log_var_high_freq = self.fc_var_high(high_freq_encoded)
        z_high_freq = self.reparameterize(mu_high_freq, log_var_high_freq)

        low_freq_recon = self.decode_low_freq(z_low_freq)
        high_freq_recon = self.decode_high_freq(z_high_freq)

        reconstructed = torch.zeros_like(input)
        
        reconstructed[:, :, low_freq_indices] = low_freq_recon
        reconstructed[:, :, high_freq_indices] = high_freq_recon

        return [reconstructed, input]

    def loss_function(self, *args, **kwargs) -> dict:
        recons = args[0]
        input = args[1]

        high_freq_input = input[:, :, high_freq_indices]
        low_freq_input = input[:, :, low_freq_indices]

        high_freq_recons = recons[:, :, high_freq_indices]
        low_freq_recons = recons[:, :, low_freq_indices]
        
        recons_loss = F.mse_loss(low_freq_recons, low_freq_input)
        w_loss = self.wasserstein_loss(high_freq_recons, high_freq_input) + F.mse_loss(high_freq_recons, high_freq_input)
        loss = recons_loss + w_loss
        
        return {'Loss': loss, 'Recons_Loss': recons_loss, 'W_Loss': w_loss}
    
    def wasserstein_loss(self, x, recon_x, std_weight=1.0):
        mean_x = torch.mean(x, dim=(0, 1), keepdim=False) 
        std_x = torch.std(x, dim=(0, 1), keepdim=False)  
        mean_recon_x = torch.mean(recon_x, dim=(0, 1), keepdim=False)  
        std_recon_x = torch.std(recon_x, dim=(0, 1), keepdim=False)    
        loss = torch.mean(torch.abs(mean_x - mean_recon_x) + std_weight * torch.abs(std_x - std_recon_x))
        
        return loss


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
low_in_dim, high_in_dim = len(low_freq_indices), len(high_freq_indices)
vae = MVAE(low_in_channels=low_in_dim, 
           high_in_channels=high_in_dim, 
           latent_dim=latent_dim, 
           hidden_dims=hidden_dim, 
           high_recon_dims=high_recon_dim, 
           low_recon_dims=low_recon_dim).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
print(vae)

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

def train_vae(dataloader, vae, num_epochs):
    vae.train()
    epoch_losses = []
    fig_cnt= 1
    plt.figure(figsize=(16, 4))
    for epoch in range(num_epochs):
        total_loss = 0.0
        total_recon_loss = 0.0
        total_w_loss = 0.0
        for err, _, _, _ in dataloader:
            optimizer.zero_grad()

            input_data = err  
            recons, input = vae(input_data.to(device), high_freq_indices, low_freq_indices)

            loss_dict = vae.loss_function(recons, input)
            loss = loss_dict['Loss']
            recon_loss = loss_dict['Recons_Loss']
            w_loss = loss_dict['W_Loss']
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_recon_loss += recon_loss.item()
            total_w_loss += w_loss.item()
        
        epoch_losses.append(total_loss / len(dataloader))
        print(f"Epoch [{epoch+1}/{num_epochs}], \t\t" 
              f"Loss: {total_loss / len(dataloader):.4f} \t" 
              f"Recons_Loss: {total_recon_loss / len(dataloader):.4f} \t" 
              f"W_Loss: {total_w_loss / len(dataloader):.4f} ")

        if epoch % 50 == 0:
            data = recons[0].cpu().detach().numpy()  
            tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
            data_2d = tsne.fit_transform(data)

            plt.subplot(1, num_epochs // 50, fig_cnt)
            plt.scatter(data_2d[:, 0], data_2d[:, 1], s=10, alpha=0.7, cmap='viridis')
            plt.title("t-SNE Visualization of Data", fontsize=10)
            plt.xlabel("t-SNE Dimension 1")
            plt.ylabel("t-SNE Dimension 2")
            plt.grid(True)
            fig_cnt += 1
    
    plt.figure(figsize=(4, 4))
    plt.plot(epoch_losses, label='Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE Training Loss Curve')
    plt.legend()
    plt.show()

num_epochs = 250
train_vae(dataloader, vae, num_epochs)

In [ ]:
look_dim = 13

In [ ]:
def test_vae(vae, test_loader, device):
    vae.eval()
    original_data = []
    reconstructed_data = []
    latent_vectors = []

    with torch.no_grad():
        for err, raw, sim, label in test_loader:
            err = err.to(device)
            label = label.to(device)
            recon_err, input = vae(err, high_freq_indices, low_freq_indices)
            recon_err = recon_err.cpu()
            label = label.cpu()
            err = err.cpu()
            
            if is_norm:
                raw_min_vals = label_ana[str(label[0][0].numpy())][0]
                raw_max_vals = label_ana[str(label[0][0].numpy())][1]
                sim_min_vals = label_ana[str(label[0][0].numpy())][2]
                sim_max_vals = label_ana[str(label[0][0].numpy())][3]
                err_min_vals = label_ana[str(label[0][0].numpy())][4]
                err_max_vals = label_ana[str(label[0][0].numpy())][5]

                err_denorm = denormalize_data(err, err_min_vals, err_max_vals)
                recon_x_denorm = denormalize_data(recon_err, err_min_vals, err_max_vals) 

                raw_denorm = denormalize_data(raw, raw_min_vals, raw_max_vals)
                sim_denorm = denormalize_data(sim, sim_min_vals, sim_max_vals)

                sim_recon_denorm = sim_denorm - recon_x_denorm
            else:
                err_denorm = err
                recon_x_denorm = recon_err
                raw_denorm = raw
                sim_denorm = sim
                sim_recon_denorm = sim_denorm - recon_x_denorm
             
            original_data.append(raw_denorm.cpu())
            reconstructed_data.append(sim_recon_denorm.cpu())

    original_data = torch.cat(original_data, dim=0)
    reconstructed_data = torch.cat(reconstructed_data, dim=0)

    plot_reconstruction(original_data, reconstructed_data)

def plot_reconstruction(original_data, reconstructed_data, dim=look_dim):
    plt.figure(figsize=(12, 6))
    seq_len = original_data.size(1)
    
    for i in range(4):
        plt.subplot(2, 2, i + 1)
        plt.plot(range(seq_len), original_data[i][:,dim].numpy(), label="Original")
        plt.plot(range(seq_len), reconstructed_data[i][:,dim].numpy(), label="Reconstructed")
        plt.legend()
        plt.title(f"Sample {i+1}")
    
    plt.tight_layout()
    plt.show()

test_vae(vae, dataloader, device)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

vae.eval()  # 设置为评估模式，以防止 dropout 等训练专用的操作

reconstructions = []
originals = []

with torch.no_grad():
    for err, raw, sim, label in dataloader:
        err = err.to(device)
        label = label.to(device)
        recon_err, input = vae(err, high_freq_indices, low_freq_indices) 
        label = label.cpu()
        recon_err = recon_err.cpu()
        err = err.cpu()
        
        if is_norm:
            raw_min_vals = label_ana[str(label[0][0].numpy())][0]
            raw_max_vals = label_ana[str(label[0][0].numpy())][1]
            sim_min_vals = label_ana[str(label[0][0].numpy())][2]
            sim_max_vals = label_ana[str(label[0][0].numpy())][3]
            err_min_vals = label_ana[str(label[0][0].numpy())][4]
            err_max_vals = label_ana[str(label[0][0].numpy())][5]

            err_denorm = denormalize_data(err, err_min_vals, err_max_vals)
            recon_x_denorm = denormalize_data(recon_err, err_min_vals, err_max_vals) 

            raw_denorm = denormalize_data(raw, raw_min_vals, raw_max_vals)
            sim_denorm = denormalize_data(sim, sim_min_vals, sim_max_vals)

            sim_recon_denorm = sim_denorm - recon_x_denorm
        else:
            err_denorm = err
            recon_x_denorm = recon_err
            raw_denorm = raw
            sim_denorm = sim
            sim_recon_denorm = sim_denorm - recon_x_denorm

        originals.append(raw_denorm.cpu())
        reconstructions.append(sim_recon_denorm.cpu())

reconstructed_data = torch.cat(reconstructions, dim=0)
original_data = torch.cat(originals, dim=0)

original_data_np = original_data.numpy()
reconstructed_data_np = reconstructed_data.numpy()

rmse_list = []
mae_list = []
r2_list = []

for i in range(original_data_np.shape[0]):
    original_batch = original_data_np[i]  # shape: (1000, 18)
    reconstructed_batch = reconstructed_data_np[i]  # shape: (1000, 18)
    
    mse_batch = mean_squared_error(original_batch, reconstructed_batch)
    rmse_batch = np.sqrt(mse_batch)
    mae_batch = mean_absolute_error(original_batch, reconstructed_batch)
    r2_batch = r2_score(original_batch, reconstructed_batch)
    
    rmse_list.append(rmse_batch)
    mae_list.append(mae_batch)
    r2_list.append(r2_batch)
    
avg_rmse = np.mean(rmse_list)
avg_mae = np.mean(mae_list)
avg_r2 = np.mean(r2_list)

print(f"Reconstruction Errors (averaged over batches):")
print(f"Average RMSE: {avg_rmse:.4f}")
print(f"Average MAE: {avg_mae:.4f}")
print(f"Average R²: {avg_r2:.4f}")


In [ ]:
import shutil

ReSaved = True

mode_name = 'multi_channel.pth'
folder_name = mode_name.split('.')[0]
folder_path = os.path.join(os.getcwd(), folder_name)
path = os.path.join(os.getcwd(), folder_name, mode_name)

if not os.path.exists(folder_path):
    os.makedirs(folder_path, exist_ok=True)

torch.save(vae, path)
print(f'Model Re-Saved to {path}')